Este es un avance de Proyecto

In [ ]:
# BLOQUE 1: INSTALACIONES E IMPORTS
!pip install prophet plotly -q
# -q = quiet (menos texto de instalación)

import pandas as pd
import numpy as np
from prophet import Prophet
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
# Silencia advertencias innecesarias

print("✅ Librerías listas")

✅ Librerías listas


In [ ]:
# BLOQUE 2: FUNCIÓN QUE LIMPIA DATOS AUTOMÁTICAMENTE
# Una "función" = bloque de código reutilizable
# def = define una función
# nombre_función(parámetros):

def limpiar_datos(df):
    """
    Limpia cualquier CSV que reciba el cliente.
    Recibe: DataFrame con columnas 'ds' y 'y'
    Devuelve: DataFrame limpio listo para Prophet
    """
    print("🔄 Limpiando datos...")

    # 1. Copiar para no modificar original
    df = df.copy()

    # 2. Convertir fechas al formato correcto
    df['ds'] = pd.to_datetime(df['ds'])
    # to_datetime = convierte texto a fecha real

    # 3. Ordenar por fecha (más antiguo primero)
    df = df.sort_values('ds').reset_index(drop=True)

    # 4. Eliminar filas con valores vacíos
    filas_antes = len(df)
    df = df.dropna(subset=['ds', 'y'])
    filas_despues = len(df)
    if filas_antes > filas_despues:
        print(f"   ⚠️ Se eliminaron {filas_antes - filas_despues} filas vacías")

    # 5. Eliminar duplicados de fecha
    df = df.drop_duplicates(subset=['ds'])

    # 6. Convertir 'y' a número
    df['y'] = pd.to_numeric(df['y'], errors='coerce')
    # errors='coerce' = si no puede convertir, pone NaN
    df = df.dropna(subset=['y'])

    # 7. Eliminar valores negativos (ventas no pueden ser negativas)
    df = df[df['y'] >= 0]

    print(f"✅ Datos limpios: {len(df)} registros válidos")
    print(f"   Desde: {df['ds'].min().date()}")
    print(f"   Hasta: {df['ds'].max().date()}")

    return df

print("✅ Función de limpieza lista")

✅ Función de limpieza lista


In [ ]:
# BLOQUE 3: FUNCIÓN QUE ENTRENA Y PREDICE
# Esta función recibe datos limpios y devuelve predicción

def predecir(df, dias_futuro=30):
    """
    Entrena Prophet y predice días futuros.
    Recibe: DataFrame limpio, días a predecir
    Devuelve: modelo entrenado, predicción, métricas
    """
    print(f"\n🔄 Entrenando modelo...")

    # 1. DIVIDIR datos (70% train, 30% test)
    split = int(len(df) * 0.70)
    df_train = df[:split]
    df_test = df[split:]

    # 2. CREAR Y ENTRENAR modelo
    modelo = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=len(df) > 365,
        # yearly solo si hay más de 1 año de datos
        daily_seasonality=False,
        interval_width=0.95
    )
    modelo.fit(df_train)

    # 3. VALIDAR con datos ocultos
    futuro_val = modelo.make_future_dataframe(
        periods=len(df_test), freq='D'
    )
    pred_val = modelo.predict(futuro_val)
    pred_test = pred_val['yhat'].tail(len(df_test)).values
    real_test = df_test['y'].values

    # 4. CALCULAR métricas
    mae = mean_absolute_error(real_test, pred_test)
    rmse = np.sqrt(mean_squared_error(real_test, pred_test))
    mape = np.mean(
        np.abs((real_test - pred_test) / real_test)
    ) * 100

    metricas = {
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2),
        'Precision': round(100 - mape, 2)
    }

    print(f"✅ Modelo validado:")
    print(f"   MAE:       {metricas['MAE']}")
    print(f"   RMSE:      {metricas['RMSE']}")
    print(f"   MAPE:      {metricas['MAPE']}%")
    print(f"   Precisión: {metricas['Precision']}%")

    # 5. REENTRENAR con TODOS los datos
    modelo_final = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=len(df) > 365,
        daily_seasonality=False,
        interval_width=0.95
    )
    modelo_final.fit(df)
    # Ahora usa 100% de datos para mejor predicción

    # 6. PREDECIR futuro real
    futuro_final = modelo_final.make_future_dataframe(
        periods=dias_futuro, freq='D'
    )
    prediccion_final = modelo_final.predict(futuro_final)

    print(f"✅ Predicción generada: {dias_futuro} días futuros")

    return modelo_final, prediccion_final, metricas

print("✅ Función de predicción lista")

✅ Función de predicción lista


In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
# BLOQUE 4: FUNCIÓN QUE GENERA DASHBOARD HTML

def generar_dashboard(df, prediccion, metricas,
                      nombre_negocio="Mi Negocio"):
    """
    Genera dashboard HTML profesional.
    Recibe: datos, predicción, métricas, nombre del negocio
    Devuelve: archivo HTML listo para cliente
    """
    print(f"\n🔄 Generando dashboard para {nombre_negocio}...")

    # ============================================
    # GRÁFICO 1: Predicción principal
    # ============================================
    fig1 = go.Figure()

    # Datos históricos
    fig1.add_trace(go.Scatter(
        x=df['ds'], y=df['y'],
        name='Ventas reales',
        line=dict(color='#2196F3', width=2),
        mode='lines+markers',
        marker=dict(size=4)
    ))

    # Predicción
    fig1.add_trace(go.Scatter(
        x=prediccion['ds'], y=prediccion['yhat'],
        name='Predicción',
        line=dict(color='#FF5722', width=2, dash='dash')
    ))

    # Intervalo confianza
    fig1.add_trace(go.Scatter(
        x=pd.concat([
            prediccion['ds'],
            prediccion['ds'][::-1]
        ]),
        y=pd.concat([
            prediccion['yhat_upper'],
            prediccion['yhat_lower'][::-1]
        ]),
        fill='toself',
        fillcolor='rgba(255,87,34,0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        name='Intervalo 95%'
    ))

    # Línea "Hoy"
    fig1.add_vline(
        x=df['ds'].max().timestamp() * 1000, # Convertir a Unix timestamp en milisegundos
        line_dash='dot',
        line_color='green',
        line_width=2,
        annotation_text='Hoy',
        annotation_position='top right'
    )

    # Caja con métricas
    fig1.add_annotation(
        x=0.02, y=0.98,
        xref='paper', yref='paper',
        text=(f"Precisión: {metricas['Precision']}%<br>"
              f"MAE: {metricas['MAE']}<br>"
              f"MAPE: {metricas['MAPE']}%" if 'MAPE' in metricas else f"MAE: {metricas['MAE']}"),
        showarrow=False,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#2196F3',
        borderwidth=1,
        font=dict(size=11)
    )


    fig1.update_layout(
        title=dict(
            text=f'{nombre_negocio} - Predicción de Ventas',
            font=dict(size=20), x=0.5
        ),
        xaxis_title='Fecha',
        yaxis_title='Ventas',
        template='plotly_white',
        hovermode='x unified',
        height=500
    )

    # ============================================
    # GRÁFICO 2: Días de la semana
    # ============================================
    dias_nombres = ['Domingo', 'Lunes', 'Martes',
                    'Miércoles', 'Jueves', 'Viernes', 'Sábado']

    weekly = prediccion[['ds', 'weekly']].copy()
    weekly['dia'] = weekly['ds'].dt.dayofweek
    weekly_avg = weekly.groupby('dia')['weekly'].mean()

    # Reindexar para que empiece en domingo
    orden = [6, 0, 1, 2, 3, 4, 5]
    valores = [weekly_avg.get(i, 0) for i in orden]

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=dias_nombres,
        y=valores,
        marker_color=[
            '#FF5722' if v == max(valores) else '#2196F3'
            for v in valores
        ],
        name='Variación por día'
    ))

    fig2.add_hline(y=0, line_dash='dash', line_color='gray')

    mejor_dia = dias_nombres[valores.index(max(valores))]

    fig2.update_layout(
        title=dict(
            text=f'¿Qué días vende más {nombre_negocio}?'
                 f' → Mejor día: {mejor_dia}',
            font=dict(size=16), x=0.5
        ),
        xaxis_title='Día de la semana',
        yaxis_title='Variación en ventas',
        template='plotly_white',
        height=400
    )

    # ============================================
    # GRÁFICO 3: Tendencia de crecimiento
    # ============================================
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(
        x=prediccion['ds'],
        y=prediccion['trend'],
        name='Tendencia',
        line=dict(color='#4CAF50', width=3)
    ))

    fig3.update_layout(
        title=dict(
            text=f'Tendencia de Crecimiento - {nombre_negocio}',
            font=dict(size=16), x=0.5
        ),
        xaxis_title='Fecha',
        yaxis_title='Ventas',
        template='plotly_white',
        height=400
    )

    # ============================================
    # EXPORTAR: Combinar en un HTML
    # ============================================
    nombre_archivo = f'/content/dashboard_{nombre_negocio.replace(" ", "_")}.html'

    with open(nombre_archivo, 'w', encoding='utf-8') as f:
        f.write(f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Dashboard - {nombre_negocio}</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background: #f5f5f5;
            margin: 0;
            padding: 20px;
        }}
        .header {{
            background: #2196F3;
            color: white;
            padding: 20px;
            border-radius: 10px;
            text-align: center;
            margin-bottom: 20px;
        }}
        .metricas {{
            display: flex;
            gap: 15px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }}
        .metrica-card {{
            background: white;
            padding: 15px 25px;
            border-radius: 10px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
            text-align: center;
            flex: 1;
            min-width: 120px;
        }}
        .metrica-valor {{
            font-size: 28px;
            font-weight: bold;
            color: #2196F3;
        }}
        .metrica-label {{
            font-size: 13px;
            color: #666;-
            margin-top: 5px;
        }}
        .grafico {{
            background: white;
            border-radius: 10px;
            padding: 15px;
            margin-bottom: 20px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }}
        .footer {{
            text-align: center;
            color: #999;
            font-size: 12px;
            margin-top: 20px;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>📊 Dashboard Predictivo de Ventas</h1>
        <h2>{nombre_negocio}</h2>
        <p>Generado con Prophet + Python</p>
    </div>

    <div class="metricas">
        <div class="metrica-card">
            <div class="metrica-valor">{metricas['Precision']}%</div>
            <div class="metrica-label">Precisión del modelo</div>
        </div>
        <div class="metrica-card">
            <div class="metrica-valor">{metricas['MAPE']}%</div>
            <div class="metrica-label">Error promedio</div>
        </div>
        <div class="metrica-card">
            <div class="metrica-valor">{mejor_dia}</div>
            <div class="metrica-label">Mejor día de venta</div>
        </div>
        <div class="metrica-card">
            <div class="metrica-valor">{len(df)}</div>
            <div class="metrica-label">Días analizados</div>
        </div>
    </div>

    <div class="grafico">
        {fig1.to_html(full_html=False, include_plotlyjs='cdn')}
    </div>

    <div class="grafico">
        {fig2.to_html(full_html=False, include_plotlyjs=False)}
    </div>

    <div class="grafico">
        {fig3.to_html(full_html=False, include_plotlyjs=False)}
    </div>

    <div class="footer">
        <p>Dashboard generado automáticamente con Prophet + Python</p>
        <p>Modelo entrenado con {len(df)} días de datos históricos</p>
    </div>
</body>
</html>
        """)

    print(f"✅ Dashboard guardado: {nombre_archivo}")
    return nombre_archivo

In [ ]:
# BLOQUE 5: FUNCIÓN PRINCIPAL QUE UNE TODO

def analizar_negocio(datos, nombre_negocio="Mi Negocio",
                     dias_futuro=30):
    """
    FUNCIÓN PRINCIPAL.
    Recibe datos de cualquier negocio y genera dashboard.

    USO:
    analizar_negocio(df, "Tienda XYZ", dias_futuro=60)
    """
    print("=" * 45)
    print(f"  ANÁLISIS: {nombre_negocio}")
    print("=" * 45)

    # PASO 1: Limpiar datos
    df_limpio = limpiar_datos(datos)

    # PASO 2: Predecir
    modelo, prediccion, metricas = predecir(
        df_limpio, dias_futuro
    )

    # PASO 3: Generar dashboard
    archivo = generar_dashboard(
        df_limpio, prediccion, metricas, nombre_negocio
    )

    print("\n" + "=" * 45)
    print("  ✅ ANÁLISIS COMPLETO")
    print("=" * 45)
    print(f"  Negocio:   {nombre_negocio}")
    print(f"  Precisión: {metricas['Precision']}%")
    print(f"  MAPE:      {metricas['MAPE']}%")
    print(f"  Dashboard: {archivo}")
    print("=" * 45)

    return df_limpio, prediccion, metricas

print("✅ Función principal lista")

✅ Función principal lista


In [ ]:
# BLOQUE 6: EJECUTAR CON DATOS DE EJEMPLO
# Simulamos datos de una tienda real

np.random.seed(42)
fechas = pd.date_range(
    start='2024-01-01', periods=90, freq='D'
)

# Datos realistas de una tienda
tendencia = 100 + np.arange(90) * 1.5
estacionalidad = 30 * np.sin(
    np.arange(90) * 2 * np.pi / 7
)
ruido = np.random.normal(0, 8, 90)
ventas = tendencia + estacionalidad + ruido

# Crear DataFrame con columnas correctas
datos_tienda = pd.DataFrame({
    'ds': fechas,
    'y': ventas
})

# ============================================
# EJECUTAR ANÁLISIS COMPLETO
# ============================================
df_resultado, prediccion, metricas = analizar_negocio(
    datos=datos_tienda,
    nombre_negocio="Tienda El Alto",
    dias_futuro=30
)

  ANÁLISIS: Tienda El Alto
🔄 Limpiando datos...
✅ Datos limpios: 90 registros válidos
   Desde: 2024-01-01
   Hasta: 2024-03-30

🔄 Entrenando modelo...
✅ Modelo validado:
   MAE:       7.02
   RMSE:      8.6
   MAPE:      3.41%
   Precisión: 96.59%
✅ Predicción generada: 30 días futuros

🔄 Generando dashboard para Tienda El Alto...
✅ Dashboard guardado: /content/dashboard_Tienda_El_Alto.html

  ✅ ANÁLISIS COMPLETO
  Negocio:   Tienda El Alto
  Precisión: 96.59%
  MAPE:      3.41%
  Dashboard: /content/dashboard_Tienda_El_Alto.html
